# 识别模型 YOLO 训练

本 Notebook 用于完成以下流程：抽帧、样本标注、导出 YOLO 数据集、训练模型、导出 ONNX。

类别（固定）: banpick, loading, gaming, victory_or_defeat, ending, other

In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src-tauri").exists() and (REPO_ROOT.parent / "src-tauri").exists():
    REPO_ROOT = REPO_ROOT.parent
TRAIN_DIR = REPO_ROOT / "train"
if str(TRAIN_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_DIR))

import utils

In [2]:
# 路径配置（按需覆盖）
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src-tauri").exists() and (REPO_ROOT.parent / "src-tauri").exists():
    REPO_ROOT = REPO_ROOT.parent
DATASET_ROOT = REPO_ROOT / "test_videos" / "recognition_dataset"
VIDEO_DIR = DATASET_ROOT / "recog_video"
STAGING_DIR = DATASET_ROOT / "staging"
MANIFEST_PATH = STAGING_DIR / "manifest.json"
YOLO_DIR = DATASET_ROOT / "yolo_dataset"
MODEL_OUT = REPO_ROOT / "models" / "yolo_game.onnx"

FRAME_DIR = STAGING_DIR / "images"
utils.ensure_dir(STAGING_DIR)
utils.ensure_dir(FRAME_DIR)

print("REPO_ROOT:", REPO_ROOT)
print("VIDEO_DIR:", VIDEO_DIR)
print("STAGING_DIR:", STAGING_DIR)
print("YOLO_DIR:", YOLO_DIR)
print("MODEL_OUT:", MODEL_OUT)

REPO_ROOT: D:\Desktop\happy\bili-shadowreplay
VIDEO_DIR: D:\Desktop\happy\bili-shadowreplay\test_videos\recognition_dataset\recog_video
STAGING_DIR: D:\Desktop\happy\bili-shadowreplay\test_videos\recognition_dataset\staging
YOLO_DIR: D:\Desktop\happy\bili-shadowreplay\test_videos\recognition_dataset\yolo_dataset
MODEL_OUT: D:\Desktop\happy\bili-shadowreplay\models\yolo_game.onnx


In [3]:
# 环境检查
def _check_cmd(cmd):
    try:
        out = subprocess.check_output(cmd, stderr=subprocess.STDOUT, text=True)
        return True, out.splitlines()[0] if out else ""
    except Exception as e:
        return False, str(e)

ok_ffmpeg, msg_ffmpeg = _check_cmd(["ffmpeg", "-version"])
print("ffmpeg 可用:", ok_ffmpeg, "|", msg_ffmpeg)

try:
    from ultralytics import YOLO
    print("ultralytics 可用")
except Exception as e:
    print("ultralytics 导入失败:", e)

try:
    import torch
    print("torch CUDA 可用:", torch.cuda.is_available())
except Exception as e:
    print("torch 不可用或初始化失败:", e)

ffmpeg 可用: True | ffmpeg version 8.0.1-essentials_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
ultralytics 可用
torch CUDA 可用: False


## 1) 抽帧并构建/合并 manifest

此单元会创建或更新 staging/manifest.json。
样本 ID 由视频路径 + 时间戳确定性生成，可避免重复。

In [ ]:
MAX_SAMPLES_PER_VIDEO = 60
DEFAULT_PHASE = "other"

manifest = utils.load_manifest(MANIFEST_PATH)
manifest["source_dir"] = str(VIDEO_DIR)
manifest["output_dir"] = str(DATASET_ROOT)
existing_ids = {s["id"] for s in manifest.get("samples", [])}

videos = utils.list_videos(VIDEO_DIR)
print("视频数量:", len(videos))

new_samples = []
for video in videos:
    # 这里按样本数量进行均匀采样；如果你能拿到真实时长，建议替换为真实时长采样。
    # 当前先用固定时长窗口（如 1800 秒）做近似采样。
    # 可按需改为 ffprobe 等方式获取真实 duration。
    duration_sec = 1800
    points = utils.build_uniform_sample_points(duration_sec, MAX_SAMPLES_PER_VIDEO)
    stem = video.stem
    out_dir = FRAME_DIR / stem
    utils.ensure_dir(out_dir)
    for sec in points:
        sample_id = utils.hash_sample_id(video, sec)
        if sample_id in existing_ids:
            continue
        frame_path = out_dir / f"{stem}_{sec}.png"
        if not frame_path.exists():
            try:
                utils.ffmpeg_extract_frame(video, sec, frame_path)
            except Exception as e:
                print("ffmpeg 抽帧失败:", video, sec, e)
                continue
        sample = utils.build_sample(
            sample_id=sample_id,
            image_path=frame_path,
            video_path=video,
            timestamp_sec=sec,
            phase=DEFAULT_PHASE,
            confidence=0.6,
            label_status="pending",
        )
        new_samples.append(sample)
        existing_ids.add(sample_id)

manifest.setdefault("samples", []).extend(new_samples)
utils.save_manifest(MANIFEST_PATH, manifest)
print("新增样本数:", len(new_samples))
print("样本总数:", len(manifest["samples"]))

## 2) 标注 / 校验界面（ipywidgets）

使用按钮为每个样本设置类别和 label_status。
修改会写回到 manifest.json。

In [ ]:
manifest = utils.load_manifest(MANIFEST_PATH)
samples = manifest.get("samples", [])

idx = 0

img_out = widgets.Output()
status_out = widgets.Output()

phase_buttons = [widgets.Button(description=name) for name in utils.CLASS_NAMES]
status_buttons = [
    widgets.Button(description="correct"),
    widgets.Button(description="wrong"),
    widgets.Button(description="pending"),
]
next_btn = widgets.Button(description="next")
prev_btn = widgets.Button(description="prev")
save_btn = widgets.Button(description="save")

def _show_sample(i):
    img_out.clear_output()
    status_out.clear_output()
    if i < 0 or i >= len(samples):
        with status_out:
            print("索引超出范围")
        return
    s = samples[i]
    path = Path(s["image_path"])
    with img_out:
        if path.exists():
            img = Image.open(path)
            plt.figure(figsize=(6, 4))
            plt.imshow(img)
            plt.axis("off")
            plt.show()
        else:
            print("图片缺失:", path)
    with status_out:
        print(f"[{i+1}/{len(samples)}] id={s['id']} phase={s['phase']} status={s['label_status']}")

def _set_phase(name):
    def _handler(_):
        s = samples[idx]
        s["phase"] = name
        s["class_id"] = utils.PHASE_TO_CLASS_ID.get(name, 5)
        s["roi"] = utils.PHASE_DEFAULT_ROI.get(name, utils.PHASE_DEFAULT_ROI["other"])
        _show_sample(idx)
    return _handler

def _set_status(name):
    def _handler(_):
        s = samples[idx]
        s["label_status"] = name
        _show_sample(idx)
    return _handler

for b in phase_buttons:
    b.on_click(_set_phase(b.description))
for b in status_buttons:
    b.on_click(_set_status(b.description))

def _next(_):
    global idx
    idx = min(idx + 1, len(samples) - 1)
    _show_sample(idx)

def _prev(_):
    global idx
    idx = max(idx - 1, 0)
    _show_sample(idx)

def _save(_):
    manifest["samples"] = samples
    utils.save_manifest(MANIFEST_PATH, manifest)
    with status_out:
        print("已保存")

next_btn.on_click(_next)
prev_btn.on_click(_prev)
save_btn.on_click(_save)

display(widgets.HBox([prev_btn, next_btn, save_btn]))
display(widgets.HBox(phase_buttons))
display(widgets.HBox(status_buttons))
display(img_out)
display(status_out)

_show_sample(idx)

## 3) 导出 YOLO 数据集

该步骤会生成 images/labels/train|val 以及 dataset.yaml。

In [4]:
import importlib
importlib.reload(utils)

manifest = utils.load_manifest(MANIFEST_PATH)
stats = utils.export_yolo_dataset(manifest, YOLO_DIR)
print(stats)
print("dataset.yaml 路径:", YOLO_DIR / "dataset.yaml")

{'images': 0, 'labels': 0, 'train_images': 0, 'val_images': 0}
dataset.yaml 路径: D:\Desktop\happy\bili-shadowreplay\test_videos\recognition_dataset\yolo_dataset\dataset.yaml


## 4) 训练 YOLO 并导出 ONNX

按需调整 epochs/imgsz。

In [5]:
from ultralytics import YOLO

EPOCHS = 3
IMGSZ = 512
RUNS_DIR = YOLO_DIR.parent / "runs"
DATA_YAML = YOLO_DIR / "dataset.yaml"
if not DATA_YAML.exists():
    raise FileNotFoundError(f"未找到数据集配置: {DATA_YAML}，请先运行第 10 个单元导出数据集")
if not (YOLO_DIR / "images" / "train").exists() or not (YOLO_DIR / "images" / "val").exists():
    raise FileNotFoundError("YOLO 图像目录缺失，请先运行第 3 个单元导出数据集")

train_count = len(list((YOLO_DIR / "images" / "train").glob("*")))
val_count = len(list((YOLO_DIR / "images" / "val").glob("*")))
print("训练集图片数:", train_count, "验证集图片数:", val_count)

model = YOLO("yolov8n.pt")

train_kwargs = dict(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=4,
    workers=0,
    amp=False,
    cache=False,
    device="cpu",
    project=str(RUNS_DIR),
    name="recognition_yolo",
    exist_ok=True,
)

try:
    model.train(**train_kwargs)
except RuntimeError as e:
    if "not enough memory" not in str(e).lower():
        raise
    print("检测到内存不足，自动降级为 batch=2, imgsz=416 重试...")
    train_kwargs["batch"] = 2
    train_kwargs["imgsz"] = 416
    model.train(**train_kwargs)

# 导出 ONNX
best_pt = RUNS_DIR / "recognition_yolo" / "weights" / "best.pt"
if not best_pt.exists():
    raise FileNotFoundError(best_pt)

exported = YOLO(str(best_pt)).export(format="onnx", imgsz=train_kwargs["imgsz"])
print("导出结果:", exported)

训练集图片数: 103 验证集图片数: 12
Ultralytics 8.4.32  Python-3.10.16 torch-2.11.0+cpu CPU (AMD Ryzen 9 7940H w/ Radeon 780M Graphics)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Desktop\happy\bili-shadowreplay\test_videos\recognition_dataset\yolo_dataset\dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=recognition_yo

## 5) 将 ONNX 复制到应用 models 目录

In [6]:
onnx_candidates = [
    YOLO_DIR.parent / "runs" / "recognition_yolo" / "weights" / "best.onnx",
    YOLO_DIR.parent / "runs" / "recognition_yolo" / "best.onnx",
]
src = next((p for p in onnx_candidates if p.exists()), None)
if src is None:
    raise FileNotFoundError("best.onnx not found")

utils.ensure_dir(MODEL_OUT.parent)
shutil.copy2(src, MODEL_OUT)
print("已复制到:", MODEL_OUT)

已复制到: D:\Desktop\happy\bili-shadowreplay\models\yolo_game.onnx


## 6) 训练集随机抽样可视化（10 张）

从训练集中随机选取 10 张图片，展示真实标签与模型预测标签，快速观察分类效果。

In [12]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from PIL import Image
from IPython.display import display
from ultralytics import YOLO

# 可调参数
SAMPLE_N = 10
CONF_THRES = 0.01  # 先降阈值，避免大量 no_det
IOU_THRES = 0.5

# 训练集路径与权重路径
train_img_dir = YOLO_DIR / "images" / "train"
train_label_dir = YOLO_DIR / "labels" / "train"

best_pt_path = RUNS_DIR / "recognition_yolo" / "weights" / "best.pt"
last_pt_path = RUNS_DIR / "recognition_yolo" / "weights" / "last.pt"

if last_pt_path.exists():
    infer_weight = last_pt_path
elif best_pt_path.exists():
    infer_weight = best_pt_path
else:
    raise FileNotFoundError(f"未找到可用于推理的权重: {last_pt_path} / {best_pt_path}")

if not train_img_dir.exists():
    raise FileNotFoundError(f"训练集目录不存在: {train_img_dir}")

all_imgs = sorted([p for p in train_img_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}])
if not all_imgs:
    raise RuntimeError(f"训练集为空: {train_img_dir}")

sample_n = min(SAMPLE_N, len(all_imgs))
sampled_imgs = random.sample(all_imgs, sample_n)

# 类别名映射（与训练时一致）
class_names = list(utils.CLASS_NAMES)

def yolo_to_xyxy(cx, cy, w, h, img_w, img_h):
    x1 = (cx - w / 2.0) * img_w
    y1 = (cy - h / 2.0) * img_h
    x2 = (cx + w / 2.0) * img_w
    y2 = (cy + h / 2.0) * img_h
    return x1, y1, x2, y2

def read_gt(img_path: Path):
    label_file = train_label_dir / f"{img_path.stem}.txt"
    if not label_file.exists():
        return None, None, "missing_label"
    lines = label_file.read_text(encoding="utf-8").strip().splitlines()
    if not lines:
        return None, None, "empty_label"

    parts = lines[0].split()
    if len(parts) < 5:
        return None, None, "bad_label"

    try:
        cls_id = int(float(parts[0]))
        cx, cy, w, h = [float(x) for x in parts[1:5]]
        if 0 <= cls_id < len(class_names):
            gt_name = class_names[cls_id]
        else:
            gt_name = f"unknown_{cls_id}"
        return cls_id, (cx, cy, w, h), gt_name
    except Exception:
        return None, None, "bad_label"

infer_model = YOLO(str(infer_weight))
records = []

print(f"抽样图片来源目录: {train_img_dir}")
print(f"使用权重: {infer_weight}")
print(f"推理阈值: conf={CONF_THRES}, iou={IOU_THRES}")
print(f"\n将展示 {sample_n} 张样本（每张图包含 GT 与 Pred）\n")

for idx, img_path in enumerate(sampled_imgs, start=1):
    img = Image.open(img_path).convert("RGB")
    img_w, img_h = img.size

    gt_cls_id, gt_box_yolo, gt_name = read_gt(img_path)

    result = infer_model.predict(
        source=str(img_path),
        imgsz=IMGSZ,
        conf=CONF_THRES,
        iou=IOU_THRES,
        verbose=False,
    )[0]

    if result.boxes is None or len(result.boxes) == 0:
        pred_name = "no_det"
        pred_conf = 0.0
        pred_xyxy = None
    else:
        confs = result.boxes.conf.cpu().numpy()
        cls_ids = result.boxes.cls.cpu().numpy().astype(int)
        xyxy = result.boxes.xyxy.cpu().numpy()
        best_idx = int(confs.argmax())

        pred_cls_id = int(cls_ids[best_idx])
        pred_conf = float(confs[best_idx])
        pred_xyxy = xyxy[best_idx]
        pred_name = class_names[pred_cls_id] if 0 <= pred_cls_id < len(class_names) else f"unknown_{pred_cls_id}"

    ok = pred_name == gt_name
    records.append(
        {
            "image": img_path.name,
            "gt": gt_name,
            "pred": pred_name,
            "pred_conf": round(pred_conf, 4),
            "match": bool(ok),
        }
    )

    fig, ax = plt.subplots(1, 1, figsize=(8, 4.8))
    ax.imshow(img)
    ax.axis("off")

    # 画 GT 框（绿色）
    if gt_box_yolo is not None:
        gx1, gy1, gx2, gy2 = yolo_to_xyxy(*gt_box_yolo, img_w, img_h)
        gt_rect = patches.Rectangle((gx1, gy1), gx2 - gx1, gy2 - gy1, linewidth=2, edgecolor="lime", facecolor="none")
        ax.add_patch(gt_rect)
        ax.text(gx1, max(0, gy1 - 6), f"GT: {gt_name}", color="lime", fontsize=10, backgroundcolor="black")

    # 画 Pred 框（红色）
    if pred_xyxy is not None:
        px1, py1, px2, py2 = pred_xyxy
        pred_rect = patches.Rectangle((px1, py1), px2 - px1, py2 - py1, linewidth=2, edgecolor="red", facecolor="none")
        ax.add_patch(pred_rect)
        ax.text(px1, min(img_h - 2, py2 + 14), f"Pred: {pred_name} ({pred_conf:.3f})", color="red", fontsize=10, backgroundcolor="white")

    title_color = "green" if ok else "red"
    ax.set_title(f"[{idx}/{sample_n}] {img_path.name}\nGT={gt_name} | Pred={pred_name} | conf={pred_conf:.3f}", color=title_color)
    plt.tight_layout()
    plt.show()

# 汇总统计 + 保存拼图
result_df = pd.DataFrame(records).sort_values(by=["match", "pred_conf"], ascending=[True, False])
correct = int(result_df["match"].sum())
acc = correct / sample_n if sample_n else 0.0

print("预测明细表:")
display(result_df)
print(f"\n抽样准确率: {acc:.2%} ({correct}/{sample_n})")

# 额外输出：生成一个简易拼图文件路径（方便你在文件管理器直接查看）
thumb_cols = 5
thumb_rows = (sample_n + thumb_cols - 1) // thumb_cols
fig, axes = plt.subplots(thumb_rows, thumb_cols, figsize=(4 * thumb_cols, 3.2 * thumb_rows))
if thumb_rows == 1:
    axes = [axes] if thumb_cols == 1 else axes

for i in range(thumb_rows * thumb_cols):
    ax = axes[i // thumb_cols][i % thumb_cols] if thumb_rows > 1 else axes[i % thumb_cols]
    ax.axis("off")
    if i >= sample_n:
        continue
    p = sampled_imgs[i]
    im = Image.open(p).convert("RGB")
    row = result_df[result_df["image"] == p.name].iloc[0]
    ax.imshow(im)
    c = "green" if bool(row["match"]) else "red"
    ax.set_title(f"GT:{row['gt']}\nPred:{row['pred']}", color=c, fontsize=9)

plt.tight_layout()
vis_out = RUNS_DIR / "recognition_yolo" / "sample_vis_10.png"
vis_out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(vis_out, dpi=150, bbox_inches="tight")
plt.show()
print(f"拼图已保存: {vis_out}")

抽样图片来源目录: D:\Desktop\happy\bili-shadowreplay\test_videos\recognition_dataset\yolo_dataset\images\train
使用权重: D:\Desktop\happy\bili-shadowreplay\test_videos\recognition_dataset\runs\recognition_yolo\weights\last.pt
推理阈值: conf=0.01, iou=0.5

将展示 10 张样本（每张图包含 GT 与 Pred）



C:\Users\Taxes\AppData\Local\Temp\ipykernel_25340\2470743707.py:141: UserWarning: Glyph 31070 (\N{CJK UNIFIED IDEOGRAPH-795E}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\Taxes\AppData\Local\Temp\ipykernel_25340\2470743707.py:141: UserWarning: Glyph 20020 (\N{CJK UNIFIED IDEOGRAPH-4E34}) missing from font(s) DejaVu Sans.
  plt.tight_layout()


<Figure size 800x480 with 1 Axes>

<Figure size 800x480 with 1 Axes>

<Figure size 800x480 with 1 Axes>

...

预测明细表:


,image,gt,pred,pred_conf,match
0,[134673][1774266510452][神临][2026-03-23_22-58-0...,gaming,loading,0.0127,False
2,[134673][1774266510452][神临][2026-03-23_22-58-0...,gaming,loading,0.0109,False
1,[134673][1774266510452][神临][2026-03-23_22-58-0...,victory_or_defeat,loading,0.0105,False
4,[134673][1774266510452][神临][2026-03-23_22-58-0...,gaming,loading,0.0103,False
3,[134673][1774266510452][神临][2026-03-23_22-58-0...,banpick,no_det,0.0000,False
5,[28915142953][1774065639769][巅峰第一五万场绝活司空震游龙思路殴...,loading,no_det,0.0000,False
6,[28915142953][1774065639769][巅峰第一五万场绝活司空震游龙思路殴...,loading,no_det,0.0000,False
7,[134673][1774266510452][神临][2026-03-23_22-58-0...,victory_or_defeat,no_det,0.0000,False
8,[28915142953][1774065639769][巅峰第一五万场绝活司空震游龙思路殴...,loading,no_det,0.0000,False
9,[28915142953][1774065639769][巅峰第一五万场绝活司空震游龙思路殴...,banpick,no_det,0.0000,False



抽样准确率: 0.00% (0/10)


<Figure size 2000x640 with 10 Axes>

拼图已保存: D:\Desktop\happy\bili-shadowreplay\test_videos\recognition_dataset\runs\recognition_yolo\sample_vis_10.png


## 7) 模型是否可以“再次训练”（无需从零开始）

可以。这里给你两种方式：

1. 断点续训（推荐）：基于 `last.pt` + `resume=True`，会继续上次训练状态（包含优化器状态）。
2. 增量微调：基于 `best.pt` 再训练新轮次，不是从随机初始化开始，但优化器状态会重置。

下面代码会优先尝试断点续训；如果找不到 `last.pt`，自动改为基于 `best.pt` 的增量微调。

In [ ]:
from ultralytics import YOLO

# 你希望“再训练”多少轮（在已有模型基础上继续）
EXTRA_EPOCHS = 10

resume_last = RUNS_DIR / "recognition_yolo" / "weights" / "last.pt"
resume_best = RUNS_DIR / "recognition_yolo" / "weights" / "best.pt"

if resume_last.exists():
    print(f"检测到 last.pt，执行断点续训: {resume_last}")
    # 断点续训：继续之前的训练状态（含优化器、学习率调度器等）
    resumed_model = YOLO(str(resume_last))
    resumed_model.train(resume=True)
elif resume_best.exists():
    print(f"未找到 last.pt，改为基于 best.pt 增量微调: {resume_best}")
    # 增量微调：从已有较好权重继续学习，但训练状态（优化器等）会重建
    finetune_model = YOLO(str(resume_best))
    finetune_model.train(
        data=str(DATA_YAML),
        epochs=EXTRA_EPOCHS,
        imgsz=IMGSZ,
        batch=4,
        workers=0,
        amp=False,
        cache=False,
        device="cpu",
        project=str(RUNS_DIR),
        name="recognition_yolo",
        exist_ok=True,
    )
else:
    raise FileNotFoundError(
        f"未找到可继续训练的权重，请先至少完成一次训练。\nlast: {resume_last}\nbest: {resume_best}"
    )

print("再次训练完成。可重新执行导出 ONNX 单元更新模型文件。")